In [174]:
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np


bikes = pd.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bikes.csv')
bikes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112475 entries, 0 to 112474
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   dteday        112475 non-null  object 
 1   hr            112475 non-null  float64
 2   casual        112475 non-null  int64  
 3   registered    112475 non-null  int64  
 4   temp_c        112475 non-null  float64
 5   feels_like_c  112475 non-null  float64
 6   hum           112475 non-null  float64
 7   windspeed     112475 non-null  float64
 8   weathersit    112475 non-null  int64  
 9   season        112475 non-null  int64  
 10  holiday       112475 non-null  int64  
 11  workingday    112475 non-null  int64  
dtypes: float64(5), int64(6), object(1)
memory usage: 10.3+ MB


In [175]:
bikes.head()

,dteday,hr,casual,registered,temp_c,feels_like_c,hum,windspeed,weathersit,season,holiday,workingday
0,1/1/2011,0.0,3,13,3.0,3.0,0.7957,0.8,1,1,0,0
1,1/1/2011,1.0,8,30,1.7,1.7,0.8272,0.8,1,1,0,0
2,1/1/2011,2.0,5,26,1.9,1.9,0.8157,1.1,1,1,0,0
3,1/1/2011,3.0,3,9,2.5,2.5,0.7831,0.8,1,1,0,0
4,1/1/2011,4.0,0,1,2.0,2.0,0.8075,1.1,1,1,0,0


In [176]:
#clean date
bikes["dteday"] = pd.to_datetime(bikes["dteday"])
bikes["year"] = bikes["dteday"].dt.year
bikes["month"] = bikes["dteday"].dt.month
bikes["day_of_week"] = bikes["dteday"].dt.dayofweek
bikes["post_covid"] = (bikes["year"] >= 2021).astype(int)


#cyclical encoding
bikes["hr_sin"] = np.sin(2 * np.pi * bikes["hr"] / 24)
bikes["hr_cos"] = np.cos(2 * np.pi * bikes["hr"] / 24)

#make target
bikes["count"] = bikes["casual"] + bikes["registered"]

# drop extra columns
bikes = bikes.drop(columns=["dteday", "hr", "casual", "registered"])

# Hot encode
bikes = pd.get_dummies(bikes, columns=["weathersit", "season"], drop_first=True)


In [177]:
bikes.head()

,temp_c,feels_like_c,hum,windspeed,holiday,workingday,year,month,day_of_week,post_covid,hr_sin,hr_cos,count,weathersit_2,weathersit_3,weathersit_4,season_2,season_3,season_4
0,3.0,3.0,0.7957,0.8,0,0,2011,1,5,0,0.000000,1.000000,16,False,False,False,False,False,False
1,1.7,1.7,0.8272,0.8,0,0,2011,1,5,0,0.258819,0.965926,38,False,False,False,False,False,False
2,1.9,1.9,0.8157,1.1,0,0,2011,1,5,0,0.500000,0.866025,31,False,False,False,False,False,False
3,2.5,2.5,0.7831,0.8,0,0,2011,1,5,0,0.707107,0.707107,12,False,False,False,False,False,False
4,2.0,2.0,0.8075,1.1,0,0,2011,1,5,0,0.866025,0.500000,1,False,False,False,False,False,False


In [178]:
bikes["year"].value_counts()

,count
year,
2012,8783
2020,8783
2016,8783
2011,8759
2013,8759
2015,8759
2014,8759
2018,8759
2017,8759


In [179]:
# split data
X = bikes.drop(columns=["count"])
y = bikes["count"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42
)

In [180]:
#log transform

y_train_log = np.log1p(y_train)
y_val = np.log1p(y_val)

In [181]:
#scale features

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [182]:
#build network

model = keras.Sequential([
    keras.Input(shape=(X_train.shape[1],)),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(128, activation="relu"),
    layers.Dense(1)
])

In [183]:
#compile model

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

In [184]:
#add early stopping
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [185]:
#train
history = model.fit(
    X_train,
    y_train_log,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop]
)


#convert predictions back to normal
preds_log = model.predict(X_test)
preds = np.expm1(preds_log)

rmse = np.sqrt(mean_squared_error(y_test, preds))
print("Test RMSE:", rmse)


Epoch 1/100
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - loss: 1.4274 - mae: 0.7805 - val_loss: 0.2390 - val_mae: 0.3368
Epoch 2/100
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - loss: 0.2802 - mae: 0.3874 - val_loss: 0.1888 - val_mae: 0.2877
Epoch 3/100
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.2278 - mae: 0.3434 - val_loss: 0.1863 - val_mae: 0.2890
Epoch 4/100
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - loss: 0.1993 - mae: 0.3161 - val_loss: 0.1860 - val_mae: 0.2675
Epoch 5/100
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.1831 - mae: 0.3017 - val_loss: 0.1954 - val_mae: 0.3144
Epoch 6/100
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - loss: 0.1755 - mae: 0.2922 - val_loss: 0.1707 - val_mae: 0.2649
Epoch 7/100
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.1715 - mae: 0.2851 - val_loss: 0.1840 - val_mae: 0.2761
Epoch 8/100
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - loss: 0.1664 - mae: 0.2826 - val_loss: 0.1756 - val_mae: 0.2837
Epoch 9/100
2250/2250 ━━━━━━━━━━

In [186]:
baseline_pred = y_train.mean()

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        np.full_like(y_test, baseline_pred)
    )
)

print("Baseline RMSE:", baseline_rmse)

Baseline RMSE: 342.88286884452964
